### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="thyroid_discordant",
    dataset_year="1986",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5D010",
    download_description="""
We download the dis.data from the UCI files. All other files are corrupted or versions of the same data as far as we know (see the dataset curation sheet for more information).

wget https://archive.ics.uci.edu/static/public/102/thyroid+disease.zip && unzip thyroid+disease.zip  dis.data dis.test && rm thyroid+disease.zip && mkdir -p local-data-warehouse/thyroid_discordant && mv dis.data dis.test local-data-warehouse/thyroid_discordant/
""",
    # References, we cite this version as we found a reference to this dataset inside of it.
    # We did not find a PDF for the other reference.
    academic_reference_bibtex="""@article{quinlan1987simplifying,
  title={Simplifying decision trees},
  author={Quinlan, J. Ross},
  journal={International journal of man-machine studies},
  volume={27},
  number={3},
  pages={221--234},
  year={1987},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="quinlan1987simplifying",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the .data file from UCI and add columns and encode nan values.

- Note, the data contains NaN indicators by default (e.g. TSH_measured). We keep them.
- We drop/ignore the referral_source columns as it indicates the original of the data and thus might leak information or makes the model learn sub-group related behavior that is not the target/task of interest.
- We correct the label column by removing the patient IDs.
- We drop 60 duplicated rows and remove constant columns.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="discordant",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="discordant",
)

## Preprocessing

In [2]:
import pandas as pd

columns = [
    "age","sex","on_thyroxine","query_on_thyroxine","on_antithyroid_medication","sick","pregnant","thyroid_surgery","I131_treatment","query_hypothyroid","query_hyperthyroid","lithium","goitre","tumor","hypopituitary","psych","TSH_measured","TSH","T3_measured","T3","TT4_measured","TT4","T4U_measured","T4U","FTI_measured","FTI","TBG_measured","TBG","referral_source","discordant",
]
df = pd.read_csv(dataset_mold.path / "dis.data", header=None, names=columns, na_values=["?"])
df = pd.concat([df, pd.read_csv(dataset_mold.path / "dis.test", header=None, names=columns, na_values=["?"])], ignore_index=True)
print("Loaded data shape:", df.shape)

df["discordant"] = df["discordant"].str.split(".").str[0] # remove patient ID from target column

as_cat_type = [
    "discordant","sex","on_thyroxine","query_on_thyroxine","on_antithyroid_medication","thyroid_surgery","query_hypothyroid","query_hyperthyroid","pregnant","sick","tumor","lithium","goitre","TSH_measured","T3_measured","TT4_measured","T4U_measured","FTI_measured","TBG_measured", "psych", "I131_treatment", "hypopituitary",
]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=[
    "referral_source", # irrelevant here
    "TBG_measured", "TBG", # constant
])

# drop duplicated rows
df = df.drop_duplicates().reset_index(drop=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (3772, 30)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 3,711
Columns: 27
Use sampling: False (sample size: 3,711)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['TSH', 'TT4', 'FTI', 'T4U', 'age', 'T3', 'sick', 'on_antithyroid_medication', 'sex', 'on_thyroxine']
Rows remaining as candidates after top-10 filter: 32 (of 3,711)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,query_hyperthyroid,lithium,goitre,tumor,hypopituitary,psych,TSH_measured,TSH,T3_measured,T3,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,discordant
0,11.0,F,f,f,t,f,f,f,f,f,f,f,f,f,f,f,f,NaN,f,NaN,f,NaN,f,NaN,f,NaN,negative
1,19.0,M,f,f,f,f,f,f,f,f,f,f,f,f,f,f,t,0.250,t,1.9,t,165.0,t,0.95,t,174.0,negative
2,59.0,F,f,f,f,f,f,f,f,f,f,f,f,f,f,f,t,1.200,f,NaN,t,140.0,t,1.63,t,86.0,negative
3,73.0,F,f,f,f,f,f,f,f,f,t,f,f,t,f,f,t,0.025,t,1.5,t,77.0,t,0.94,t,82.0,negative
4,33.0,F,f,f,f,f,t,f,f,f,t,f,f,f,f,f,t,3.000,t,2.5,t,121.0,t,1.38,t,88.0,negative


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,sex,category,149.0,4.02,2.0,"F, M"
1,on_thyroxine,category,0.0,0.00,2.0,"f, t"
2,query_on_thyroxine,category,0.0,0.00,2.0,"f, t"
3,on_antithyroid_medication,category,0.0,0.00,2.0,"f, t"
4,sick,category,0.0,0.00,2.0,"f, t"
5,pregnant,category,0.0,0.00,2.0,"f, t"
6,thyroid_surgery,category,0.0,0.00,2.0,"f, t"
7,I131_treatment,category,0.0,0.00,2.0,"f, t"
8,query_hypothyroid,category,0.0,0.00,2.0,"f, t"
9,query_hyperthyroid,category,0.0,0.00,2.0,"f, t"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,3710.0,51.860377,20.123585,1.000,455.00
TSH,3402.0,5.087820,24.524998,0.005,530.00
T3,3002.0,2.013504,0.827572,0.050,10.60
TT4,3540.0,108.328475,35.605132,2.000,430.00
T4U,3384.0,0.994989,0.195485,0.250,2.32
FTI,3386.0,110.480715,33.088316,2.000,395.00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                    rank                          
FTI_measured              1              t   3386  91.24
                          2              f    325   8.76
I131_treatment            1              f   3652  98.41
                          2              t     59   1.59
T3_measured               1              t   3002  80.89
                          2              f    709  19.11
T4U_measured              1              t   3384  91.19
                          2              f    327   8.81
TSH_measured              1              t   3402  91.67
                          2              f    309   8.33
TT4_measured              1              t   3540  95.39
                          2              f    171   4.61
discordant                1       negative   3653  98.44
                          2     discordant     58   1.56
goitre                    1              f   3677  99.08
                          2              t     34   0.92
hypopituitary             1              f   3710  99.97
                          2              t      1   0.03
lithium                   1              f   3693  99.51
                          2              t     18   0.49
on_antithyroid_medication 1              f   3669  98.87
                          2              t     42   1.13
on_thyroxine              1              f   3247  87.50
                          2              t    464  12.50
pregnant                  1              f   3658  98.57
                          2              t     53   1.43
psych                     1              f   3527  95.04
                          2              t    184   4.96
query_hyperthyroid        1              f   3477  93.69
                          2              t    234   6.31
query_hypothyroid         1              f   3477  93.69
                          2              t    234   6.31
query_on_thyroxine        1              f   3661  98.65
                          2              t     50   1.35
sex                       1              F   2424  65.32
                          2              M   1138  30.67
                          3           <NA>    149   4.02
sick                      1              f   3564  96.04
                          2              t    147   3.96
thyroid_surgery           1              f   3658  98.57
                          2              t     53   1.43
tumor                     1              f   3615  97.41
                          2              t     96   2.59

In [8]:
# Target Distribution
target_df

,count,pct
discordant,,
negative,3653,98.44
discordant,58,1.56


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to thyroid_discordant/019d5dca-624f-7cfe-a990-252bb72c7d6b


019d5dca-624f-7cfe-a990-252bb72c7d6b
0d625fed09ab4b2e590ac1771d01efdb31ee43905cf0ae0968ae7351deb01d1c
